<a href="https://colab.research.google.com/github/mrivassnj-svg/AI-Digital-Twins/blob/main/A05_06_TeamYYT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# APP 3 (CJ) - NetQ-style Network Monitor with "What Just Happened" (WJH)
# Concept: agents stream telemetry to a central platform; WJH explains WHY packets drop;
# separate NETWORK problems from EQUIPMENT/APP problems; translate into business impact.
# Not real NetQ - a teaching simulation. Colab:  !pip install -q gradio
# ============================================================
import numpy as np, pandas as pd
from matplotlib.figure import Figure
import gradio as gr

DUR, CAP = 60, 100.0   # seconds simulated, uplink capacity (Mbps)
# name: (baseline Mbps, priority (1=highest), switch port, business criticality 1-3, consequence)
SERVICES = {"Scoring system": (2, 1, "swp1", 3, "Scores stop updating; lanes can't be started/billed"),
            "Payment terminals": (1, 1, "swp2", 3, "Customers can't pay; queue at front desk"),
            "Arcade card system": (1, 2, "swp3", 2, "Cards can't be loaded or swiped"),
            "Security cameras": (40, 3, "swp4", 2, "Gaps in recorded footage"),
            "Guest Wi-Fi": (30, 3, "swp5", 1, "Poor guest experience (low criticality)")}
SCENARIOS = ["Normal operation", "Guest Wi-Fi surge (congestion)", "Scoring port link down (physical)",
             "ACL misconfig blocks payments", "Bad cable on cameras (physical)", "Arcade no-route (routing)",
             "Scoring app crash (NOT a network fault)"]

def simulate(scenario, qos, wjh_ok, seed=3):
    rng = np.random.default_rng(int(seed)); names = list(SERVICES); n = len(names); ix = {k: i for i, k in enumerate(names)}
    base = np.array([SERVICES[k][0] for k in names], float); prio = np.array([SERVICES[k][1] for k in names])
    produced = np.clip(base[:, None] * (1 + .08 * rng.standard_normal((n, DUR))), 0, None)   # what equipment generates
    sent = produced.copy(); drop = np.zeros((n, DUR)); why = np.full((n, DUR), "", dtype=object)
    def hit(name, a, b, rate, reason): drop[ix[name], a:b] = rate; why[ix[name], a:b] = reason
    if scenario.startswith("Guest"): produced[ix["Guest Wi-Fi"], 20:50] = sent[ix["Guest Wi-Fi"], 20:50] = 110
    elif scenario.startswith("Scoring port"): hit("Scoring system", 25, 45, 1.0, "Physical: link down")
    elif scenario.startswith("ACL"): hit("Payment terminals", 30, DUR, 1.0, "ACL: blocked by rule")
    elif scenario.startswith("Bad cable"): hit("Security cameras", 20, DUR, .15, "Physical: CRC / signal errors")
    elif scenario.startswith("Arcade"): hit("Arcade card system", 30, 50, 1.0, "Routing: no route to destination")
    elif scenario.startswith("Scoring app"): produced[ix["Scoring system"], 25:45] = sent[ix["Scoring system"], 25:45] = 0

    deliv, cong, lat = np.zeros((n, DUR)), np.zeros((n, DUR)), np.zeros(DUR)
    for t in range(DUR):
        d = sent[:, t] * (1 - drop[:, t]); tot = d.sum()
        if tot > CAP:
            if qos:                                              # strict priority queuing protects critical services
                out, left = np.zeros(n), CAP
                for p in sorted(set(prio)):
                    m = prio == p; need = d[m].sum()
                    out[m] = d[m] if need <= left else d[m] * left / need; left = max(0, left - need)
            else: out = d * CAP / tot                            # everyone suffers proportionally
            cong[:, t] = d - out; d = out
        deliv[:, t] = d; lat[t] = 2 + (180 if tot > CAP else 60 * max(0, d.sum() / CAP - .75))

    UNK = "Unknown - switch not WJH-capable (counters only)"
    ev = {}
    for i, k in enumerate(names):
        for t in range(DUR):
            if sent[i, t] > .01 and drop[i, t] > 0: ev.setdefault((k, why[i, t] if wjh_ok else UNK), []).append((t, sent[i, t] * drop[i, t]))
            if cong[i, t] > .05 * max(sent[i, t], .01): ev.setdefault((k, "Congestion: buffer overflow (uplink full)" if wjh_ok else UNK), []).append((t, cong[i, t]))
    events = pd.DataFrame([dict(Service=k, Port=SERVICES[k][2], **{"What Just Happened": r}, **{"First (s)": v[0][0], "Last (s)": v[-1][0],
                            "Seconds": len(v), "Peak drop (Mbps)": round(max(x for _, x in v), 1)}) for (k, r), v in ev.items()])
    if events.empty: events = pd.DataFrame([dict(Service="-", Port="-", **{"What Just Happened": "No drops reported"})])

    health, diag, score, conseq = [], [], 0, []
    for i, k in enumerate(names):
        lost = (sent[i] - deliv[i]).sum(); collapse = produced[i].min() < .2 * base[i]
        bad = int((deliv[i] < .7 * base[i]).sum()); score += bad * SERVICES[k][3]
        health.append((k, SERVICES[k][2], round(100 * deliv[i].sum() / max(produced[i].sum(), 1e-9), 1), round(lost, 1)))
        if collapse and lost < .02 * max(base[i] * DUR, 1):
            diag.append(f"- 🛠️ **{k}**: link is up, **zero drops**, but the equipment stopped producing data → **equipment/app fault**, not the network.")
        elif lost > .02 * sent[i].sum():
            rs = sorted({r for (kk, r) in ev if kk == k})
            diag.append(f"- 🌐 **{k}**: equipment is producing data but it's being lost in transit → **network fault** - " + ("; ".join(rs)))
        if bad: conseq.append((k, bad, SERVICES[k][4]))
    sev = "None" if score == 0 else "Minor" if score < 60 else "Moderate" if score < 200 else "Major"
    if not diag: diag = ["- ✅ All services healthy; telemetry matches equipment output."]
    if not wjh_ok and any("🌐" in d for d in diag): diag.append("- ℹ️ Without WJH you can see *that* packets dropped, not *why*; more manual digging is needed.")

    fig = Figure(figsize=(9, 6)); a1, a2 = fig.subplots(2, 1, sharex=True); T = np.arange(DUR)
    a1.plot(T, sent.sum(0), label="Offered load"); a1.plot(T, deliv.sum(0), label="Delivered"); a1.axhline(CAP, c="r", ls=":", label="Uplink capacity")
    a1.set_ylabel("Uplink (Mbps)"); a1.legend(loc="upper left"); a1.set_title("Telemetry from the agents")
    for i, k in enumerate(names): a2.plot(T, np.clip(100 * deliv[i] / base[i], 0, 160), label=k)
    a2.set_ylabel("% of normal throughput"); a2.set_xlabel("Time (s)"); a2.legend(fontsize=7, ncol=3, loc="lower left"); fig.tight_layout()
    imp = f"### Business impact: **{sev}** (score {score})\n" + ("\n".join(f"- {k}: {s}s degraded - {c}" for k, s, c in conseq) or "- No customer-facing impact")
    return fig, events, "\n".join(diag), imp, pd.DataFrame(health, columns=["Service", "Port", "Delivered vs produced (%)", "Total dropped (Mb)"])

with gr.Blocks(title="NetQ-style Monitor") as demo:
    gr.Markdown("# 📡 Bowling Alley Network Monitor (NetQ-style)\nAgents report telemetry to one central view. **WJH** explains *why* packets drop, "
                "and the twin translates that into business impact. *Teaching simulation - NetQ only applies to compatible NVIDIA infrastructure.*")
    with gr.Row():
        sc = gr.Dropdown(SCENARIOS, value=SCENARIOS[1], label="Scenario"); qos = gr.Checkbox(False, label="QoS priority for critical services")
        wjh = gr.Checkbox(True, label="Switch is WJH-capable (e.g., Spectrum)")
    go = gr.Button("Run monitor", variant="primary")
    plot = gr.Plot(); diag = gr.Markdown(); imp = gr.Markdown()
    ev = gr.Dataframe(label="What Just Happened (WJH) events"); ph = gr.Dataframe(label="Port health")
    ins, outs = [sc, qos, wjh], [plot, ev, diag, imp, ph]
    go.click(simulate, ins, outs); demo.load(simulate, ins, outs)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9084d2fb1d3a65e1c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#ANNETTE

In [4]:
import gradio as gr
import numpy as np
import pandas as pd
from matplotlib.figure import Figure

def simulate(protocol, hop_latency, jitter, bandwidth, loss_pct, hops, rate, payload_size):
    # 1. Generate Physical Signal (60 seconds)
    duration = 60.0
    N = int(duration * rate)
    t = np.arange(N) / rate

    # Baseline vibration (sine wave) + Pinsetter reset spikes every 20s
    baseline = np.sin(2 * np.pi * 0.5 * t)
    spikes = (10.0 * np.exp(-((t - 10)**2) / 0.05) +
              10.0 * np.exp(-((t - 30)**2) / 0.05) +
              10.0 * np.exp(-((t - 50)**2) / 0.05))
    signal = baseline + spikes

    # 2. Network Queue & Delays
    bandwidth_bps = bandwidth * 1e6
    payload_bits = payload_size * 8
    serialization_delay = payload_bits / bandwidth_bps
    base_prop_delay = hops * (hop_latency / 1000.0)

    # Calculate when packets leave the bottleneck queue
    t_leave_queue = np.zeros(N)
    q_end = 0.0
    for i in range(N):
        q_end = max(q_end, t[i]) + serialization_delay
        t_leave_queue[i] = q_end

    prop_delays = np.maximum(0, base_prop_delay + np.random.normal(0, jitter / 1000.0, N))
    drops = np.random.rand(N) < (loss_pct / 100.0)

    t_arrival = np.full(N, np.inf)
    retransmissions = 0

    # 3. Transport Protocol Logic
    if protocol == "UDP":
        valid = ~drops
        t_arrival[valid] = t_leave_queue[valid] + prop_delays[valid]

    elif protocol == "TCP":
        for i in range(N):
            t_send = t[i]
            success = not drops[i]
            rto = max(0.2, 2 * (base_prop_delay + serialization_delay))
            t_rcv = t_leave_queue[i] + prop_delays[i]

            while not success:
                retransmissions += 1
                t_send += rto
                rto = min(2.0, rto * 2)
                success = np.random.rand() >= (loss_pct / 100.0)
                t_rcv = t_send + serialization_delay + base_prop_delay + np.random.normal(0, jitter / 1000.0)
            t_arrival[i] = t_rcv

        # Enforce in-order delivery
        max_t = 0.0
        for i in range(N):
            max_t = max(max_t, t_arrival[i])
            t_arrival[i] = max_t

    # 4. Digital Twin Reconstruction
    t_eval = np.linspace(0, duration, 2000)
    twin_values = np.zeros_like(t_eval)
    staleness = np.zeros_like(t_eval)

    events = [(t_arrival[i], t[i], signal[i]) for i in range(N) if t_arrival[i] < np.inf]
    events.sort(key=lambda x: x[0])

    latest_sensor_t = -1.0
    current_val = 0.0
    event_idx = 0

    for j, te in enumerate(t_eval):
        while event_idx < len(events) and events[event_idx][0] <= te:
            arr_t, sens_t, val = events[event_idx]
            if sens_t > latest_sensor_t:
                latest_sensor_t = sens_t
                current_val = val
            event_idx += 1
        twin_values[j] = current_val
        staleness[j] = (te - latest_sensor_t) if latest_sensor_t >= 0 else te

    # 5. Calculate Metrics
    packets_delivered = len(events)
    initial_loss_count = np.sum(drops)

    delays = t_arrival[t_arrival < np.inf] - t[t_arrival < np.inf]
    mean_delay = np.mean(delays) if len(delays) > 0 else 0
    p95_delay = np.percentile(delays, 95) if len(delays) > 0 else 0

    avg_staleness = np.mean(staleness)
    twin_error = np.mean(np.abs(np.interp(t_eval, t, signal) - twin_values))

    spike_times = [10, 30, 50]
    detected_spikes = 0
    lags = []

    for st in spike_times:
        mask = (t_eval >= st - 1) & (t_eval <= st + 3)
        if np.any(twin_values[mask] > 5.0):
            detected_spikes += 1
            first_cross_t = t_eval[mask][twin_values[mask] > 5.0][0]
            true_peak_t = t[(t >= st - 1) & (t <= st + 3)][np.argmax(signal[(t >= st - 1) & (t <= st + 3)])]
            lags.append(max(0, first_cross_t - true_peak_t))

    avg_detection_lag = np.mean(lags) if lags else 0.0

    # 6. Plotting with softer, modern tech colors
    fig = Figure(figsize=(10, 6.5), layout="tight")
    ax1, ax2 = fig.subplots(2, 1, sharex=True)

    ax1.plot(t, signal, label="Physical Sensor (Ground Truth)", alpha=0.4, color='#8E9AAF', linestyle='--')
    ax1.plot(t_eval, twin_values, label=f"Digital Twin ({protocol})", color='#00E5FF', linewidth=1.8)
    ax1.set_title("Vibration Signal: Physical vs Twin Simulation", color='#E2E8F0', fontsize=11, fontweight='bold')
    ax1.set_ylabel("Amplitude", color='#94A3B8')
    ax1.legend(loc="upper right", facecolor='#1E293B', edgecolor='#00E5FF', labelcolor='white')
    ax1.grid(True, alpha=0.1, color='#94A3B8')
    ax1.set_facecolor('#0F172A')
    ax1.tick_params(colors='#94A3B8')

    ax2.plot(t_eval, staleness, color='#FF5E7E', linewidth=1.5)
    ax2.fill_between(t_eval, 0, staleness, color='#FF5E7E', alpha=0.15)
    ax2.set_title("Twin Staleness (Telemetry Gap / Desync Lag)", color='#E2E8F0', fontsize=11, fontweight='bold')
    ax2.set_xlabel("Time (seconds)", color='#94A3B8')
    ax2.set_ylabel("Staleness (s)", color='#94A3B8')
    ax2.grid(True, alpha=0.1, color='#94A3B8')
    ax2.set_facecolor('#0F172A')
    ax2.tick_params(colors='#94A3B8')
    fig.patch.set_facecolor('#1E293B')

    # 7. Metrics Table
    df_metrics = pd.DataFrame({
        "System Telemetry Parameter": [
            "Packets Dispatched",
            "Packets Successfully Delivered",
            "Original Lost Packets",
            "TCP Active Retransmissions",
            "Average Network Latency (s)",
            "Peak 95th Percentile Delay (s)",
            "Average Twin Telemetry Lag (s)",
            "Mean Reconstruction Error",
            "Pinsetter Cycles Captured",
            "Average Delay to Detect Spikes (s)"
        ],
        "Value": [
            f"{N}",
            f"{packets_delivered}",
            f"{initial_loss_count} ({initial_loss_count/N*100:.1f}%)",
            f"{retransmissions}" if protocol == "TCP" else "Disabled (UDP Flow)",
            f"{mean_delay:.4f}s",
            f"{p95_delay:.4f}s",
            f"{avg_staleness:.4f}s",
            f"{twin_error:.4f}",
            f"{detected_spikes} out of 3 resets",
            f"{avg_detection_lag:.4f}s"
        ]
    })

    # 8. Friendly Insights with clear translations
    insights = []
    if np.max(t_leave_queue - t) > 1.0:
        insights.append("⚠️ **Queue Congestion Warning:** The selected bandwidth isn't wide enough to handle the streaming rate. Telemetry data is waiting in a growing queue, introducing permanent delay.")
    elif avg_staleness > 1.0:
        insights.append("⚠️ **Out-of-Sync Twin:** The digital representation has high lag. This happens due to delivery delays, high packet losses, or TCP wait cycles.")
    elif loss_pct > 0 and protocol == "UDP":
        insights.append("💡 **UDP Behavior Profile:** By skipping error corrections, UDP avoids waiting blocks and maintains low lag (staleness). However, you'll see a slightly rugged wave due to lost points.")
    elif loss_pct > 0 and protocol == "TCP":
        insights.append("💡 **TCP Behavior Profile:** TCP guarantees delivery via retry rules. However, when packets drop, the twin temporarily pauses (visible as peaks on the staleness graph) until correct data arrives.")

    if detected_spikes < 3:
        insights.append("❌ **Data Fidelity Alert:** The twin missed some high-speed pinsetter events. This could lead to late or missed diagnostic predictions.")

    if not insights:
        insights.append("✅ **Perfect Link Alignment:** Excellent bandwidth and zero packet loss. The virtual twin perfectly represents the real physical system instantly.")

    return fig, df_metrics, "\n\n".join(insights)

def load_preset(name):
    presets = {
        "Healthy LAN": ["TCP", 1.0, 0.1, 100.0, 0.0, 1, 50, 100],
        "Congested Link": ["TCP", 5.0, 5.0, 0.05, 1.0, 3, 50, 500],
        "Lossy Cable (TCP)": ["TCP", 2.0, 1.0, 10.0, 10.0, 2, 50, 200],
        "Lossy Cable (UDP)": ["UDP", 2.0, 1.0, 10.0, 10.0, 2, 50, 200],
        "Far-away Cloud Twin": ["TCP", 35.0, 10.0, 50.0, 2.0, 12, 50, 200]
    }
    return presets[name]

# Custom styling for high-tech aesthetic but clean layout
neon_theme = gr.themes.Soft(
    primary_hue="cyan",
    secondary_hue="slate",
    neutral_hue="slate",
).set(
    body_background_fill="#111827",
    body_background_fill_dark="#111827",
    button_primary_background_fill="#06B6D4",
    button_primary_background_fill_hover="#0891B2",
    button_primary_text_color="#FFFFFF",
    block_background_fill="#1F2937",
    block_label_text_color="#22D3EE",
    block_title_text_color="#22D3EE"
)

with gr.Blocks(title="Bowling Alley Digital Twin") as demo:
    gr.HTML("""
    <div style='text-align: center; padding: 20px; background: linear-gradient(135deg, #1e1b4b, #111827); border-radius: 12px; border: 1px solid #22d3ee; margin-bottom: 25px;'>
        <h1 style='color: #22d3ee; text-shadow: 0 0 10px rgba(34, 211, 238, 0.5); font-weight: 800; font-size: 2rem; margin: 0;'>
            ⚡ Bowling Alley Digital Twin
        </h1>
        <p style='color: #94a3b8; font-size: 1rem; margin-top: 8px;'>
            Interactive Network Twin Simulator — Physical Vibration Telemetry & Virtual Mapping
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🛠️ Step 1: Set Scenario or Configure Network")
            preset_cb = gr.Dropdown(
                choices=["Healthy LAN", "Congested Link", "Lossy Cable (TCP)", "Lossy Cable (UDP)", "Far-away Cloud Twin"],
                label="⚡ Select Environment Preset",
                value="Healthy LAN"
            )

            with gr.Accordion("⚙️ Customize Core Network Variables", open=True):
                protocol_rb = gr.Radio(["TCP", "UDP"], label="Transport Protocol", value="TCP")
                with gr.Row():
                    latency_sl = gr.Slider(0.1, 100.0, value=1.0, label="Hop Latency (ms)")
                    jitter_sl = gr.Slider(0.0, 20.0, value=0.1, label="Jitter (ms)")
                with gr.Row():
                    bw_sl = gr.Slider(0.01, 1000.0, value=100.0, label="Bandwidth (Mbps)")
                    loss_sl = gr.Slider(0.0, 50.0, value=0.0, label="Packet Loss (%)")

            with gr.Accordion("⚙️ Customize Edge Sensor Parameters", open=False):
                with gr.Row():
                    hops_sl = gr.Slider(1, 20, step=1, value=1, label="Network Hop Count")
                    rate_sl = gr.Slider(1, 200, step=1, value=50, label="Sensor Sampling (Hz)")
                payload_sl = gr.Slider(32, 1500, step=1, value=100, label="Payload Size (Bytes)")

            run_btn = gr.Button("🚀 Run Signal & Twin Simulation", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 📊 Step 2: Diagnostic Plots, Insights & Metrics")
            plot_out = gr.Plot(label="Signal Comparison Plot")

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("**💡 Digital Twin Analysis**")
                    insights_out = gr.Markdown()
                with gr.Column(scale=1):
                    gr.Markdown("**📋 System Telemetry Indicators**")
                    table_out = gr.Dataframe(headers=["System Telemetry Parameter", "Value"], interactive=False)

    preset_cb.change(
        load_preset,
        inputs=preset_cb,
        outputs=[protocol_rb, latency_sl, jitter_sl, bw_sl, loss_sl, hops_sl, rate_sl, payload_sl]
    )

    inputs_list = [protocol_rb, latency_sl, jitter_sl, bw_sl, loss_sl, hops_sl, rate_sl, payload_sl]
    run_btn.click(simulate, inputs=inputs_list, outputs=[plot_out, table_out, insights_out])
    demo.load(simulate, inputs=inputs_list, outputs=[plot_out, table_out, insights_out])

demo.launch(theme=neon_theme)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e4eaf47392ee7c5469.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#Diondria

In [6]:
import gradio as gr
import pandas as pd
import matplotlib.patches as patches
from matplotlib.figure import Figure

# --- 1. Expected Topology Definition ---
expected_p2p = [
    {"Link ID": "L1", "Src Node": "Twin Server (TS)", "Src Port": "eth0", "Dst Node": "Core Switch (CS)", "Dst Port": "port1", "Role": "Server Uplink"},
    {"Link ID": "L2", "Src Node": "Core Switch (CS)", "Src Port": "port2", "Dst Node": "Lane 1 Switch (LS1)", "Dst Port": "eth_up", "Role": "Lane 1 Uplink"},
    {"Link ID": "L3", "Src Node": "Core Switch (CS)", "Src Port": "port3", "Dst Node": "Lane 2 Switch (LS2)", "Dst Port": "eth_up", "Role": "Lane 2 Uplink"},
    {"Link ID": "L4", "Src Node": "Lane 1 Switch (LS1)", "Src Port": "eth1", "Dst Node": "Sensor 1 (S1)", "Dst Port": "eth0", "Role": "Lane 1 Sensor"},
    {"Link ID": "L5", "Src Node": "Lane 1 Switch (LS1)", "Src Port": "eth2", "Dst Node": "Pinsetter 1 (P1)", "Dst Port": "eth0", "Role": "Lane 1 Pinsetter"},
    {"Link ID": "L6", "Src Node": "Lane 1 Switch (LS1)", "Src Port": "eth3", "Dst Node": "Scorer 1 (SC1)", "Dst Port": "eth0", "Role": "Lane 1 Scorer"},
    {"Link ID": "L7", "Src Node": "Lane 2 Switch (LS2)", "Src Port": "eth1", "Dst Node": "Sensor 2 (S2)", "Dst Port": "eth0", "Role": "Lane 2 Sensor"},
    {"Link ID": "L8", "Src Node": "Lane 2 Switch (LS2)", "Src Port": "eth2", "Dst Node": "Pinsetter 2 (P2)", "Dst Port": "eth0", "Role": "Lane 2 Pinsetter"},
    {"Link ID": "L9", "Src Node": "Lane 2 Switch (LS2)", "Src Port": "eth3", "Dst Node": "Scorer 2 (SC2)", "Dst Port": "eth0", "Role": "Lane 2 Scorer"},
]

node_positions = {
    "Twin Server (TS)": (0, 4),
    "Core Switch (CS)": (0, 3),
    "Lane 1 Switch (LS1)": (-2, 2),
    "Lane 2 Switch (LS2)": (2, 2),
    "Sensor 1 (S1)": (-3, 1),
    "Pinsetter 1 (P1)": (-2, 1),
    "Scorer 1 (SC1)": (-1, 1),
    "Sensor 2 (S2)": (1, 1),
    "Pinsetter 2 (P2)": (2, 1),
    "Scorer 2 (SC2)": (3, 1)
}

# --- 2. Simulation Logic ---
def simulate_discovery(faults):
    results = []
    graph = {node: [] for node in node_positions.keys()}

    # Evaluate expected vs actual based on faults
    for link in expected_p2p:
        l_id = link["Link ID"]
        fault = faults.get(l_id, "None")

        src = link["Src Node"]
        dst = link["Dst Node"]

        if fault == "None":
            res = "PASS"
            discovered = f"{dst}:{link['Dst Port']}"
            graph[src].append(dst)
            graph[dst].append(src)
        elif fault == "Missing Cable":
            res = "MISSING"
            discovered = "NULL (No Link Detected)"
        elif fault == "Link Down":
            res = "LINK DOWN"
            discovered = f"{dst}:{link['Dst Port']} (Oper DOWN)"
        elif fault == "Miswired":
            res = "MISWIRED"
            discovered = f"UNKNOWN_DEVICE:port_x"

        results.append({
            "Link ID": l_id,
            "Expected": f"{src} -> {dst}",
            "Discovered": discovered,
            "Validation Result": res
        })

    # Reachability BFS from Twin Server
    reachable = set(["Twin Server (TS)"])
    queue = ["Twin Server (TS)"]
    while queue:
        curr = queue.pop(0)
        for neighbor in graph[curr]:
            if neighbor not in reachable:
                reachable.add(neighbor)
                queue.append(neighbor)

    return pd.DataFrame(results), reachable

def generate_outputs(faults):
    val_df, reachable = simulate_discovery(faults)

    # Topology Map
    fig = Figure(figsize=(9, 5), layout="tight")
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.set_facecolor("#1e1e2e")
    fig.patch.set_facecolor('#1e1e2e')

    # Draw Links
    for _, row in val_df.iterrows():
        l_id = row["Link ID"]
        res = row["Validation Result"]
        link_def = next(l for l in expected_p2p if l["Link ID"] == l_id)
        pos1 = node_positions[link_def["Src Node"]]
        pos2 = node_positions[link_def["Dst Node"]]

        color = "#a6e3a1" if res == "PASS" else "#f38ba8"
        linestyle = "-" if res == "PASS" else "--"
        linewidth = 2.5 if res == "PASS" else 2.0

        ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], color=color, linestyle=linestyle, linewidth=linewidth, zorder=1)
        # Link Label
        mid_x = (pos1[0] + pos2[0]) / 2
        mid_y = (pos1[1] + pos2[1]) / 2
        ax.text(mid_x, mid_y, l_id, color="white", fontsize=8, ha="center", va="center",
                bbox=dict(facecolor="#313244", edgecolor="none", boxstyle="round,pad=0.2"), zorder=2)

    # Draw Nodes
    for node, pos in node_positions.items():
        is_reachable = node in reachable
        node_color = "#89b4fa" if is_reachable else "#f38ba8"
        edge_color = "#b4befe" if is_reachable else "#eba0ac"

        circle = patches.Circle(pos, radius=0.25, facecolor=node_color, edgecolor=edge_color, linewidth=2, zorder=3)
        ax.add_patch(circle)

        label_y = pos[1] - 0.35 if "Switch" not in node and "Server" not in node else pos[1] + 0.35
        ax.text(pos[0], label_y, node.split("(")[1].replace(")",""), color="white", fontsize=10, fontweight="bold", ha="center", va="center", zorder=4)

    # Checklist
    all_pass = (val_df["Validation Result"] == "PASS").all()
    checklist = f"""
    ### 📝 Pre-Deployment Checklist
    - ✅ **Inventory Loaded**: Expected P2P file imported successfully.
    - ✅ **P2P File Consistent**: Syntactic checks passed.
    - {"✅" if all_pass else "❌"} **Access Test from Collector**: Can reach all network agents.
    - {"✅" if all_pass else "❌"} **Agents Reporting**: Discovered topology matches expected.
    """

    # Twin States
    def get_lane_state(lane_num):
        devices = [f"Sensor {lane_num} (S{lane_num})", f"Pinsetter {lane_num} (P{lane_num})", f"Scorer {lane_num} (SC{lane_num})"]
        r_count = sum(1 for d in devices if d in reachable)
        if r_count == 3: return "🟢 **Healthy** (All data streams active)"
        elif r_count > 0: return f"🟡 **Degraded** ({r_count}/3 devices reachable)"
        else: return "🔴 **Offline** (No communication with lane)"

    twin_states = f"""
    ### 🎳 Digital Twin Operational States
    - **Lane 1 Twin**: {get_lane_state(1)}
    - **Lane 2 Twin**: {get_lane_state(2)}
    """

    # Data Loss DataFrame
    loss_data = []
    for lane in [1, 2]:
        if f"Sensor {lane} (S{lane})" not in reachable:
            loss_data.append({"Lane": lane, "Lost Data Stream": "Ball Events", "Twin Impact": "Cannot track ball speed/trajectory; physics engine stalled."})
        if f"Pinsetter {lane} (P{lane})" not in reachable:
            loss_data.append({"Lane": lane, "Lost Data Stream": "Reset Confirmation", "Twin Impact": "Pin state unknown; simulation out of sync with physical deck."})
        if f"Scorer {lane} (SC{lane})" not in reachable:
            loss_data.append({"Lane": lane, "Lost Data Stream": "Scores", "Twin Impact": "Player performance metrics missing from virtual dashboard."})

    loss_df = pd.DataFrame(loss_data) if loss_data else pd.DataFrame([{"Lane": "None", "Lost Data Stream": "None", "Twin Impact": "All systems operating normally."}])

    return fig, checklist, val_df, twin_states, loss_df

# --- 3. Gradio Interface ---
def apply_fault(current_faults, link, fault_type):
    current_faults[link] = fault_type
    fig, check, val, states, loss = generate_outputs(current_faults)
    return current_faults, fig, check, val, states, loss

def clear_all():
    fig, check, val, states, loss = generate_outputs({})
    return {}, fig, check, val, states, loss

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    fault_state = gr.State({})

    gr.Markdown("# 🌐 Bowling Alley P2P Map & Validator")
    gr.Markdown("Inspired by the NVIDIA Cable Validation Tool. Validate actual network cabling against the expected Unified Point-to-Point (P2P) file. Inject faults to see how physical network issues cascade into digital twin degradation.")

    with gr.Accordion("Expected Unified P2P Table", open=False):
        gr.Dataframe(pd.DataFrame(expected_p2p))

    with gr.Row():
        link_dd = gr.Dropdown([l["Link ID"] for l in expected_p2p], label="Target Link", value="L1")
        fault_dd = gr.Dropdown(["None", "Missing Cable", "Link Down", "Miswired"], label="Inject Fault", value="None")
        inject_btn = gr.Button("Apply Fault / Update", variant="primary")
        clear_btn = gr.Button("Clear All Faults")

    with gr.Row():
        with gr.Column(scale=2):
            map_plot = gr.Plot(label="Discovered Topology & Reachability")
        with gr.Column(scale=1):
            checklist_md = gr.Markdown()
            twin_states_md = gr.Markdown()

    with gr.Row():
        val_table = gr.Dataframe(label="Cable Validation Results (Expected vs Discovered)")
        loss_table = gr.Dataframe(label="Lost Operational Data Impacts")

    # Event wiring
    inject_btn.click(
        apply_fault,
        inputs=[fault_state, link_dd, fault_dd],
        outputs=[fault_state, map_plot, checklist_md, val_table, twin_states_md, loss_table]
    )

    clear_btn.click(
        clear_all,
        inputs=[],
        outputs=[fault_state, map_plot, checklist_md, val_table, twin_states_md, loss_table]
    )

    # Initialize UI
    demo.load(clear_all, inputs=[], outputs=[fault_state, map_plot, checklist_md, val_table, twin_states_md, loss_table])

demo.launch()

/tmp/ipykernel_1761/4207230836.py:168: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://681699e3703c57ae3b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#CJ

In [7]:
import gradio as gr
import numpy as np
import pandas as pd
from matplotlib.figure import Figure

# ---------------------------------------------------------
# Core Configuration
# ---------------------------------------------------------
SERVICES = {
    "Scoring": {"base_load": 5, "priority": 1, "port": "swp1", "criticality": "Critical", "impact": "Bowling games halt, scores lost."},
    "Payments": {"base_load": 2, "priority": 1, "port": "swp2", "criticality": "Critical", "impact": "Unable to process transactions or sell food/games."},
    "Arcade Cards": {"base_load": 10, "priority": 2, "port": "swp3", "criticality": "Moderate", "impact": "Arcade games unplayable, lost revenue."},
    "Cameras": {"base_load": 40, "priority": 2, "port": "swp4", "criticality": "Moderate", "impact": "Security blindspots, liability risk."},
    "Guest Wi-Fi": {"base_load": 20, "priority": 3, "port": "swp5", "criticality": "Low", "impact": "Guests complain about slow internet."}
}

UPLINK_CAPACITY = 100.0 # Mbps

# ---------------------------------------------------------
# Simulation Engine
# ---------------------------------------------------------
def simulate_traffic(scenario, qos_enabled, wjh_enabled):
    time = np.arange(60)

    offered = {}
    delivered = {}
    drops = {}
    port_status = {s: "UP" for s in SERVICES.keys()}

    # 1. Generate Offered Load based on Scenario
    for s_name, s_info in SERVICES.items():
        base = s_info["base_load"]
        noise = np.random.uniform(-0.1 * base, 0.1 * base, 60)
        offered[s_name] = np.maximum(0, base + noise)

    if scenario == "Guest Wi-Fi Surge":
        offered["Guest Wi-Fi"] = 80 + np.random.uniform(-5, 5, 60)
    elif scenario == "Scoring App Crash":
        offered["Scoring"] = np.zeros(60) # App stopped sending, but link is up
    elif scenario == "Scoring Port Link Down":
        offered["Scoring"] = np.zeros(60)
        port_status["Scoring"] = "DOWN"

    # 2. Apply Network Rules (ACLs, Routes, Physical faults)
    drop_reasons_internal = {s: None for s in SERVICES.keys()}

    for s_name in SERVICES.keys():
        delivered[s_name] = offered[s_name].copy()

    if scenario == "ACL Blocking Payments":
        delivered["Payments"] = np.zeros(60)
        drop_reasons_internal["Payments"] = "ACL Rule Deny (Ingress)"
    elif scenario == "Arcade No-Route":
        delivered["Arcade Cards"] = np.zeros(60)
        drop_reasons_internal["Arcade Cards"] = "L3 No Route to Destination"
    elif scenario == "Bad Cable on Cameras":
        # Flapping link: drops random chunks of data
        flaps = np.random.choice([0, 1], size=60, p=[0.3, 0.7])
        delivered["Cameras"] = delivered["Cameras"] * flaps
        drop_reasons_internal["Cameras"] = "L1 FCS Error / Rx Drops"

    # 3. Handle Uplink Congestion (QoS or FIFO)
    # Calculate pre-congestion delivered
    total_intent = np.sum([delivered[s] for s in SERVICES.keys()], axis=0)

    for t in range(60):
        if total_intent[t] > UPLINK_CAPACITY:
            if qos_enabled:
                # Strict Priority: P1 > P2 > P3
                available = UPLINK_CAPACITY
                for p in [1, 2, 3]:
                    # Get services in this priority level
                    p_services = [s for s, info in SERVICES.items() if info["priority"] == p]
                    p_demand = sum(delivered[s][t] for s in p_services)

                    if p_demand <= available:
                        available -= p_demand
                    else:
                        # Share remaining equally among same priority, drop the rest
                        ratio = available / p_demand if p_demand > 0 else 0
                        for s in p_services:
                            delivered[s][t] *= ratio
                            if drop_reasons_internal[s] is None:
                                drop_reasons_internal[s] = "Egress Queue Tail Drop (Congestion)"
                        available = 0
            else:
                # FIFO / Proportional drop
                ratio = UPLINK_CAPACITY / total_intent[t]
                for s in SERVICES.keys():
                    delivered[s][t] *= ratio
                    if drop_reasons_internal[s] is None:
                        drop_reasons_internal[s] = "Egress Queue Tail Drop (Congestion)"

    # 4. Calculate Drops
    for s_name in SERVICES.keys():
        drops[s_name] = np.maximum(0, offered[s_name] - delivered[s_name])

    total_offered = np.sum([offered[s] for s in SERVICES.keys()], axis=0)
    total_delivered = np.sum([delivered[s] for s in SERVICES.keys()], axis=0)

    # 5. Build What Just Happened (WJH) Event Table
    wjh_events = []
    for s_name in SERVICES.keys():
        if np.sum(drops[s_name]) > 1.0: # Detectable drops
            d_times = np.where(drops[s_name] > 0.1)[0]
            if len(d_times) > 0:
                reason = drop_reasons_internal[s_name] if wjh_enabled else "Unknown (counters only)"
                wjh_events.append({
                    "Service": s_name,
                    "Port": SERVICES[s_name]["port"],
                    "Drop Reason": reason,
                    "First Drop (s)": d_times[0],
                    "Last Drop (s)": d_times[-1],
                    "Duration (s)": d_times[-1] - d_times[0] + 1,
                    "Peak Drop (Mbps)": round(np.max(drops[s_name]), 2)
                })

    wjh_df = pd.DataFrame(wjh_events)
    if wjh_df.empty:
        wjh_df = pd.DataFrame(columns=["Service", "Port", "Drop Reason", "First Drop (s)", "Last Drop (s)", "Duration (s)", "Peak Drop (Mbps)"])

    # 6. Build Port Health Table
    health_data = []
    for s_name in SERVICES.keys():
        pct_norm = (np.mean(delivered[s_name]) / SERVICES[s_name]["base_load"]) * 100
        health_data.append({
            "Service": s_name,
            "Port": SERVICES[s_name]["port"],
            "Admin/Oper State": port_status[s_name],
            "Throughput % (vs Normal)": round(pct_norm, 1),
            "Total Drops (Mb)": round(np.sum(drops[s_name]), 1)
        })
    health_df = pd.DataFrame(health_data)

    # 7. Generate Plots (using Figure, not pyplot)
    fig = Figure(figsize=(10, 6), layout="tight")
    ax1, ax2 = fig.subplots(1, 2)

    # Plot A: Total Uplink
    ax1.plot(time, total_offered, label="Total Offered", color='gray', linestyle='--')
    ax1.plot(time, total_delivered, label="Total Delivered", color='blue')
    ax1.axhline(UPLINK_CAPACITY, color='red', linestyle=':', label="Uplink Capacity (100Mbps)")
    ax1.set_title("Uplink Bandwidth Utilization")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Throughput (Mbps)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot B: Per-Service Health Bar Chart
    services_list = list(SERVICES.keys())
    pcts = [(np.mean(delivered[s]) / SERVICES[s]["base_load"]) * 100 for s in services_list]
    colors = ['green' if p > 90 else 'orange' if p > 50 else 'red' for p in pcts]
    ax2.bar(services_list, pcts, color=colors)
    ax2.axhline(100, color='gray', linestyle='--')
    ax2.set_title("Service Delivery (% of Normal)")
    ax2.set_ylabel("% Delivered")
    ax2.set_ylim(0, 110)
    ax2.tick_params(axis='x', rotation=45)

    # 8. Diagnosis Logic
    impact_level = "🟢 Minor"
    diagnosis = "Network is operating normally. All services are passing traffic within expected parameters."

    if scenario == "Guest Wi-Fi Surge":
        if qos_enabled:
            diagnosis = "**Diagnosis:** Network fault (Congestion). Guest Wi-Fi traffic spiked, exceeding uplink capacity. Because QoS is enabled, the switch actively dropped Low-Priority Guest Wi-Fi packets to protect Critical services (Scoring, Payments). The network is functioning exactly as designed to mitigate congestion."
            impact_level = "🟡 Moderate (Guest experience degraded, Business unaffected)"
        else:
            diagnosis = "**Diagnosis:** Network fault (Congestion). Guest Wi-Fi spiked, and without QoS, the switch's FIFO buffers dropped packets across ALL services randomly. Critical bowling and payment traffic was lost alongside guest traffic."
            impact_level = "🔴 Major (Facility-wide disruptions)"

    elif scenario == "Scoring Port Link Down":
        diagnosis = "**Diagnosis:** Physical Network Fault. The physical link on swp1 (Scoring) went DOWN. No traffic can enter or leave the port. Check the physical cable, SFP transceiver, or power on the Scoring server."
        impact_level = "🔴 Major (Scoring offline)"

    elif scenario == "ACL Blocking Payments":
        diagnosis = "**Diagnosis:** Configuration Fault. Traffic is arriving at swp2, but the switch is dropping 100% of it due to an Access Control List (ACL) rule. This is often caused by a recent firewall or policy misconfiguration."
        impact_level = "🔴 Major (Payments offline)"

    elif scenario == "Bad Cable on Cameras":
        diagnosis = "**Diagnosis:** Physical Layer Fault. The link is technically UP, but Frame Check Sequence (FCS) errors are causing the switch hardware to discard corrupted frames. This usually indicates a damaged cable or failing optic."
        impact_level = "🟡 Moderate (Camera feed glitching)"

    elif scenario == "Arcade No-Route":
        diagnosis = "**Diagnosis:** Layer 3 Routing Fault. Packets from the Arcade system enter the switch successfully, but the routing table lacks a valid next-hop destination. The switch drops the packets because it doesn't know where to send them."
        impact_level = "🟡 Moderate (Arcades offline)"

    elif scenario == "Scoring App Crash":
        diagnosis = "**Diagnosis:** **NON-NETWORK FAULT**. Notice that the Scoring port is UP, and there are ZERO network drops on the port. However, throughput is 0%. This proves the network is perfectly healthy, but the *application* (Scoring Server) has crashed and stopped sending data. Do not troubleshoot the network; reboot the server."
        impact_level = "🔴 Major (App issue)"

    # Add WJH note to diagnosis
    if not wjh_enabled and scenario != "Normal" and scenario != "Scoring App Crash" and scenario != "Scoring Port Link Down":
        diagnosis += "\n\n*Note: Without WJH enabled, we only see that drops occurred, but we have to guess WHY by manually checking ACLs, routes, and physical interfaces.*"

    return fig, wjh_df, health_df, diagnosis, impact_level


# ---------------------------------------------------------
# Gradio Interface
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("# 🎳 Bowling Alley Network Monitor (NetQ-style)")
    gr.Markdown("> ⚠️ **TEACHING SIMULATION ONLY:** This is an interactive educational model to demonstrate network monitoring concepts. Actual NVIDIA NetQ software only operates on compatible NVIDIA/Mellanox networking infrastructure.")

    with gr.Row():
        scenario_dd = gr.Dropdown(
            choices=[
                "Normal",
                "Guest Wi-Fi Surge",
                "Scoring Port Link Down",
                "ACL Blocking Payments",
                "Bad Cable on Cameras",
                "Arcade No-Route",
                "Scoring App Crash"
            ],
            value="Normal",
            label="Inject Scenario"
        )
        qos_cb = gr.Checkbox(label="Enable QoS (Prioritize Critical Services)", value=True)
        wjh_cb = gr.Checkbox(label="Switch is 'What Just Happened' (WJH) Capable", value=True)

    with gr.Row():
        plot_out = gr.Plot(label="Traffic Analysis")

    with gr.Row():
        with gr.Column(scale=1):
            impact_md = gr.Markdown("### Business Impact")
            diag_md = gr.Markdown("### Diagnosis")
        with gr.Column(scale=2):
            wjh_table = gr.Dataframe(label="What Just Happened (WJH) Drop Events")

    with gr.Row():
        health_table = gr.Dataframe(label="Port Health & Statistics")

    # Wire it up
    inputs = [scenario_dd, qos_cb, wjh_cb]
    outputs = [plot_out, wjh_table, health_table, diag_md, impact_md]

    scenario_dd.change(simulate_traffic, inputs, outputs)
    qos_cb.change(simulate_traffic, inputs, outputs)
    wjh_cb.change(simulate_traffic, inputs, outputs)

    # Init
    demo.load(simulate_traffic, inputs, outputs)

demo.launch()

/tmp/ipykernel_1761/3201145624.py:202: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Default()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74fe51b98ee76e013e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#Michael

In [1]:
# ============================================================
# APP 4 (Team) - Schematic Construction & 3D Prototype Assembler
# Generates each layer of the bowling-alley digital twin and stacks them into one 3D prototype:
#   L1 Physical lanes/equipment  L2 Cabling P2P map (Diondria)  L3 Network transport (Annette)
#   L4 Monitoring agents + WJH (CJ)  L5 Digital twin state
# Colab:  !pip install -q gradio plotly
# ============================================================
import numpy as np
import plotly.graph_objects as go
import gradio as gr

LAYERS = ["L1 Physical lanes & equipment", "L2 Cabling (P2P map)", "L3 Network transport",
          "L4 Monitoring (agents + WJH)", "L5 Digital twin"]
POS = {"TwinServer": (-8, 1.5), "Core-SW": (-5.5, 1.5), "Lane1-SW": (-3, 0), "Lane2-SW": (-3, 3),
       "L1-Scorer": (-1, 0), "L1-Sensor": (2, 0), "L1-Pinsetter": (10, 0),
       "L2-Scorer": (-1, 3), "L2-Sensor": (2, 3), "L2-Pinsetter": (10, 3)}
EDGES = [("L1-Sensor", "Lane1-SW"), ("L1-Pinsetter", "Lane1-SW"), ("L1-Scorer", "Lane1-SW"),
         ("L2-Sensor", "Lane2-SW"), ("L2-Pinsetter", "Lane2-SW"), ("L2-Scorer", "Lane2-SW"),
         ("Lane1-SW", "Core-SW"), ("Lane2-SW", "Core-SW"), ("Core-SW", "TwinServer")]
FAULTS = {"None": {}, "Lane 1 sensor cable missing": {("L1-Sensor", "Lane1-SW"): "missing"},
          "Lane 1 uplink down": {("Lane1-SW", "Core-SW"): "down"},
          "Congested uplinks (guest traffic)": {("Lane1-SW", "Core-SW"): "congested", ("Lane2-SW", "Core-SW"): "congested"}}
WJH = {"missing": "Physical: no neighbor discovered", "down": "Physical: link down", "congested": "Congestion: buffer overflow"}
EDGE_COL = {"ok": "#546e7a", "missing": "#e53935", "down": "#fb8c00", "congested": "#fdd835"}
STATE_COL = {"Healthy": "#43a047", "Degraded": "#fdd835", "Offline": "#e53935"}

def box(x0, x1, y0, y1, z0, z1, color, opacity):
    x = [x0, x1, x1, x0, x0, x1, x1, x0]; y = [y0, y0, y1, y1, y0, y0, y1, y1]; z = [z0] * 4 + [z1] * 4
    return go.Mesh3d(x=x, y=y, z=z, i=[0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3], j=[1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4],
                     k=[2, 3, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7], color=color, opacity=opacity, flatshading=True, hoverinfo="skip")

def lane_states(status):
    adj = {}
    for (a, b), s in status.items():
        if s != "missing" and s != "down": adj.setdefault(a, []).append(b); adj.setdefault(b, []).append(a)
    seen, q = {"TwinServer"}, ["TwinServer"]
    while q:
        for n in adj.get(q.pop(), []):
            if n not in seen: seen.add(n); q.append(n)
    out = {}
    for L in (1, 2):
        dev = [n for n in POS if n.startswith(f"L{L}-")] + [f"Lane{L}-SW"]
        k = sum(n not in seen for n in dev)
        cong = status.get((f"Lane{L}-SW", "Core-SW")) == "congested"
        out[L] = "Offline" if k >= 3 else "Degraded" if (k or cong) else "Healthy"
    return out

def build(step, explode, fault):
    step, gap = int(step), 0.8 + 4 * explode
    z = [i * gap for i in range(5)]
    status = {e: FAULTS[fault].get(e, "ok") for e in EDGES}
    states, tr = lane_states(status), []
    for i in range(step):                                              # layer name tags
        tr.append(go.Scatter3d(x=[-10.5], y=[-2], z=[z[i]], mode="text", text=[LAYERS[i]], textposition="middle right", showlegend=False, hoverinfo="skip"))
    if step >= 1:                                                      # L1 physical
        for y in (0, 3): tr.append(box(-0.5, 10.5, y - .8, y + .8, z[0], z[0] + .2, "#8d6e63", 1))
        d = [n for n in POS if n[0] == "L"]
        tr.append(go.Scatter3d(x=[POS[n][0] for n in d], y=[POS[n][1] for n in d], z=[z[0] + .5] * len(d), mode="markers+text",
                               text=[n.split("-")[1] for n in d], textfont=dict(size=9), marker=dict(size=5, color="#5d4037"), name="Equipment"))
    if step >= 2:                                                      # L2 cabling
        for (a, b), s in status.items():
            tr.append(go.Scatter3d(x=[POS[a][0], POS[b][0]], y=[POS[a][1], POS[b][1]], z=[z[1]] * 2, mode="lines",
                                   line=dict(color=EDGE_COL[s], width=6, dash="dash" if s in ("missing", "down") else "solid"), hoverinfo="name", name=f"{a}→{b} ({s})", showlegend=False))
        nx, ny, nz = zip(*[(p[0], p[1], z[1]) for p in POS.values()])
        tr.append(go.Scatter3d(x=nx, y=ny, z=nz, mode="markers+text", text=[n if not n[0] == "L" or "SW" in n else "" for n in POS],
                               textfont=dict(size=9), marker=dict(size=6, symbol="square", color="#263238"), name="Nodes"))
        dx, dy, dz = [], [], []
        for n in POS:
            if n[0] == "L" and "SW" not in n: dx += [POS[n][0], POS[n][0], None]; dy += [POS[n][1], POS[n][1], None]; dz += [z[0] + .2, z[1], None]
        tr.append(go.Scatter3d(x=dx, y=dy, z=dz, mode="lines", line=dict(color="#90a4ae", width=2, dash="dot"), hoverinfo="skip", showlegend=False))
    if step >= 3:                                                      # L3 transport: packets on live links
        px, py, pc = [], [], []
        for (a, b), s in status.items():
            if s in ("missing", "down"): continue
            n = 5 if s == "ok" else 12
            f = np.linspace(.1, .9, n) if s == "ok" else np.linspace(.6, .95, n)      # congestion bunches packets up
            px += list(POS[a][0] + f * (POS[b][0] - POS[a][0])); py += list(POS[a][1] + f * (POS[b][1] - POS[a][1])); pc += [s] * n
        tr.append(go.Scatter3d(x=px, y=py, z=[z[2]] * len(px), mode="markers", marker=dict(size=3.5, color=["#1e88e5" if c == "ok" else "#fb8c00" for c in pc]), name="Packets"))
    if step >= 4:                                                      # L4 monitoring
        tr.append(go.Scatter3d(x=[p[0] for p in POS.values()], y=[p[1] for p in POS.values()], z=[z[3]] * len(POS), mode="markers",
                               marker=dict(size=5, symbol="diamond", color="#8e24aa"), name="Agents"))
        al = [(e, s) for e, s in status.items() if s != "ok"]
        if al:
            tr.append(go.Scatter3d(x=[(POS[a][0] + POS[b][0]) / 2 for (a, b), _ in al], y=[(POS[a][1] + POS[b][1]) / 2 for (a, b), _ in al], z=[z[3]] * len(al),
                                   mode="markers+text", text=["WJH: " + WJH[s] for _, s in al], textposition="top center", textfont=dict(size=10, color="#c62828"),
                                   marker=dict(size=9, symbol="x", color="#e53935"), name="WJH alerts"))
    if step >= 5:                                                      # L5 digital twin
        for L, y in ((1, 0), (2, 3)):
            tr.append(box(-0.5, 10.5, y - .8, y + .8, z[4], z[4] + .3, STATE_COL[states[L]], .55))
            tr.append(go.Scatter3d(x=[5], y=[y], z=[z[4] + .6], mode="text", text=[f"Lane {L} twin: {states[L]}"], showlegend=False, hoverinfo="skip"))
    fig = go.Figure(tr)
    fig.update_layout(height=650, margin=dict(l=0, r=0, t=30, b=0), title="Bowling alley digital twin - layered prototype",
                      scene=dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False, camera=dict(eye=dict(x=1.2, y=-1.8, z=0.9))),
                      legend=dict(orientation="h"))
    md = "**Assembled layers:** " + " → ".join(LAYERS[:step]) + "\n\n" + (f"**Lane twins:** Lane 1 {states[1]} · Lane 2 {states[2]}" if step >= 5 else "_Add layer 5 to see the twin state._")
    return fig, md

with gr.Blocks(title="3D Prototype Assembler") as demo:
    gr.Markdown("# 🧱 Schematic Construction → 3D Digital Twin Prototype\nBuild the twin layer by layer, then explode the stack apart to inspect each layer. "
                "Inject a fault to see it propagate from cabling → packets → WJH alerts → twin state.")
    with gr.Row():
        step = gr.Slider(1, 5, 5, step=1, label="Build step (layers assembled)")
        explode = gr.Slider(0, 1, 0.4, label="Exploded view (0 = assembled)")
        fault = gr.Dropdown(list(FAULTS), value="None", label="Inject fault")
    plot = gr.Plot(); info = gr.Markdown(); ins = [step, explode, fault]
    for c in ins: c.change(build, ins, [plot, info])
    demo.load(build, ins, [plot, info])

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c23fbc667a07988dd9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
